# Five slice networks with fixed seeds: OASIS-1 and ADNI1 (A100 runtime)

Runs `scripts/slice_networks.py`, the seeded re-implementation of the five OASIS-1 baseline notebooks, on one cohort
(`COHORT` below) over the run matrix of `docs/prespecification_networks_adni.md`:

| cohort | models | seeds | training pool | runs |
|---|---|---|---|---|
| `oasis` | 5 | 0, 1, 2 | full (about 181 per fold) | 15 × 5 folds |
| `adni` | 5 | 0, 1, 2 | full (about 481) and 181 (learning curve) | 30 × 5 folds |

Every fold writes its out-of-fold CSV and a JSON log straight to Drive (`ADNI/networks/runs_<cohort>/`), so a
disconnected session resumes where it stopped: just run all cells again. Expected A100 time: OASIS about 6 h, ADNI
about 20 h (two or three sessions). Checkpoints stay on the local disk and are deleted after each fold.

**Needs on Drive**: for `adni`, `ADNI/adni_network_cohort.csv` and `ADNI/networks/adni_slices_fov_v1.tar` (from
`adni_slices_colab.ipynb`); for `oasis`, `alzheimer_anfis_outputs/results/oof_predictions_ANFIS-on-GT.csv` and the
slice folder `alzheimer_anfis_outputs/baseline_densenet121_slices/` (archived to `ADNI/networks/oasis_slices.tar` on
the first run).

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # must precede the first import of torch (deterministic cuBLAS)

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/oasis1-cognitive-baseline'
REPO_REF = 'networks-v1'   # git tag pinned by docs/prespecification_networks_adni.md
if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', '--quiet', 'https://github.com/Yuliawang520/oasis1-cognitive-baseline', REPO])
subprocess.check_call(['git', '-C', REPO, 'fetch', '--quiet', '--tags'])
subprocess.check_call(['git', '-C', REPO, 'checkout', '--quiet', REPO_REF])
print('repo at', subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
sys.path.insert(0, f'{REPO}/scripts')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
import torch, torchvision, timm
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| timm', timm.__version__)

In [ ]:
COHORT = 'adni'            # 'adni' or 'oasis'
DRIVE = '/content/drive/MyDrive'
MODELS = ['densenet121', 'vgg16', 'resnet50', 'deit_small', 'swin_t']
SEEDS = [0, 1, 2]
if COHORT == 'adni':
    COHORT_CSV = f'{DRIVE}/ADNI/adni_network_cohort.csv'
    SLICES_TAR = f'{DRIVE}/ADNI/networks/adni_slices_fov_v1.tar'
    SLICES_SRC = None
    POOL_SIZES = [None, 181]       # full training pool, then the OASIS-1 pool size (learning curve)
else:
    COHORT_CSV = f'{DRIVE}/alzheimer_anfis_outputs/results/oof_predictions_ANFIS-on-GT.csv'
    SLICES_TAR = f'{DRIVE}/ADNI/networks/oasis_slices.tar'
    SLICES_SRC = f'{DRIVE}/alzheimer_anfis_outputs/baseline_densenet121_slices'
    POOL_SIZES = [None]
OUT_DIR = f'{DRIVE}/ADNI/networks/runs_{COHORT}'
SLICES_LOCAL = f'/content/slices_{COHORT}'
os.makedirs(OUT_DIR, exist_ok=True)
print(COHORT, '->', OUT_DIR)

In [ ]:
# Stage the slices on the local disk (reading thousands of PNGs through the Drive mount is far slower).
import shutil, tarfile
if not os.path.isdir(SLICES_LOCAL) or not os.listdir(SLICES_LOCAL):
    if os.path.exists(SLICES_TAR):
        os.makedirs(SLICES_LOCAL, exist_ok=True)
        with tarfile.open(SLICES_TAR) as t:
            try:
                t.extractall(SLICES_LOCAL, filter='data')
            except TypeError:
                t.extractall(SLICES_LOCAL)
    elif SLICES_SRC and os.path.isdir(SLICES_SRC):
        shutil.copytree(SLICES_SRC, SLICES_LOCAL)
        os.makedirs(os.path.dirname(SLICES_TAR), exist_ok=True)
        shutil.make_archive(SLICES_TAR[:-4], 'tar', SLICES_LOCAL)
    else:
        raise FileNotFoundError(f'neither {SLICES_TAR} nor {SLICES_SRC} exists')
n_subj = len([d for d in os.listdir(SLICES_LOCAL) if os.path.isdir(f'{SLICES_LOCAL}/{d}')])
print(n_subj, 'subject folders in', SLICES_LOCAL)

In [ ]:
# Train. Finished folds (CSV present in OUT_DIR) are skipped, so this cell can be re-run after a disconnect.
from pathlib import Path
import slice_networks as SN
summary = SN.run_matrix(COHORT, Path(COHORT_CSV), Path(SLICES_LOCAL), Path(OUT_DIR), MODELS, SEEDS, POOL_SIZES,
                        work_dir=Path('/content/work'))

In [ ]:
# Seed ensembles (the pre-specified primary output) and a per-seed overview.
import pandas as pd
from sklearn.metrics import roc_auc_score
rows = []
for m in MODELS:
    for pool in POOL_SIZES:
        try:
            ens = SN.seed_ensemble(Path(OUT_DIR), COHORT, m, pool, SEEDS)
        except FileNotFoundError as e:
            print('incomplete:', m, pool or 'full', '->', e)
            continue
        ens.to_csv(f'{OUT_DIR}/ensemble_{COHORT}_{m}_pool{pool or "full"}.csv', index=False)
        y = (ens['label'] == 'Demented').astype(int)
        aucs = [roc_auc_score(y, ens[f'seed{s}']) for s in SEEDS]
        rows.append({'model': m, 'pool': pool or 'full', 'seed AUCs': [round(a, 3) for a in aucs],
                     'mean': round(sum(aucs) / len(aucs), 3), 'ensemble AUC': round(roc_auc_score(y, ens['prob_demented']), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nsend back the whole folder:', OUT_DIR)